<p style="text-align:center; font-size:14px;  margin-bottom:10px;">
  &copy; 2026 Shizhe Chen. All rights reserved.
</p>
<hr>

## What Is a Transformer?

**Transformers** are deep learning architectures that rely entirely on **attention mechanisms**—no recurrent or convolutional layers—to model relationships in sequential data.  
Initially designed for language translation, they now power state‑of‑the‑art models in NLP, vision, audio, and beyond.

---

### How It Works

1. **Token & Positional Encoding**  
   Each input token is embedded into a vector and augmented with a *positional encoding* to retain order information.

2. **Multi‑Head Self‑Attention**  
   For every position, the model attends to *all* other positions, computing weighted combinations of token representations—allowing it to capture both short‑ and long‑range dependencies.

3. **Feed‑Forward Networks & Residual Connections**  
   Position‑wise MLPs refine the representations, while residual/skip connections and layer normalization stabilize and speed up training.

4. **Stacked Layers**  
   Encoder and decoder stacks repeat the attention‑plus‑FFN block several times, enabling deep hierarchical feature learning.

---

### Pros and Cons

✅ Captures global context in a single layer  
✅ Highly parallelizable → fast training on GPUs  
✅ Scales well to very large datasets & models  

⚠️ Quadratic memory/time cost in sequence length  
⚠️ Requires large datasets to realize full potential  


In [ ]:
# -----------------------------------------------------------
# 🔬 A Minimal Scaled Dot‑Product Attention Demo
# -----------------------------------------------------------
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute attention = softmax(QK^T / sqrt(d_k)) V"""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)     # [batch, heads, seq, seq]
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)                    # attention weights
    return attn @ V, attn                               # returns context & weights

# Toy inputs: 1 batch, 2 heads, sequence length 4, d_k = 3
torch.manual_seed(42)
Q = torch.randn(1, 2, 4, 3)   # queries
K = torch.randn(1, 2, 4, 3)   # keys
V = torch.randn(1, 2, 4, 3)   # values

context, weights = scaled_dot_product_attention(Q, K, V)
print("Context shape:", context.shape)   # (1, heads=2, seq=4, d_k=3)
print("Attention weights (head 0):\n", weights[0,0])


In [ ]:
import torch
import matplotlib.pyplot as plt

def plot_attention(
    attn_weights: torch.Tensor,
    tokens: list[str],
    head: int = 0,
    batch: int = 0,
    cmap: str | None = None,
):
    """
    Visualize the attention weights for a single head as a heat-map.
    
    Parameters
    ----------
    attn_weights : Tensor
        Shape (batch, n_heads, seq_len, seq_len)
    tokens : list[str]
        The text tokens corresponding to the sequence dimension.
    head : int, default 0
        Which attention head to plot.
    batch : int, default 0
        Which batch element to plot (usually 0 for demos).
    cmap : str | None
        Optional Matplotlib colormap name. Leave None to let MPL choose its default.
    """
    # 1️⃣ Select the slice you want
    w = attn_weights[batch, head].detach().cpu().numpy()   # (seq, seq)
    
    # 2️⃣ Create the plot
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(w, aspect="auto", interpolation="nearest", cmap=cmap)
    
    # 3️⃣ Add token labels
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens)
    
    ax.set_xlabel("Key / Value positions")
    ax.set_ylabel("Query positions")
    ax.set_title(f"Head {head} attention weights")
    
    # 4️⃣ Add a color bar for scale
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

# --- Example usage with your toy data ------------------------
seq_tokens = ["q0", "q1", "q2", "q3"]  # or real words if you have them
plot_attention(weights, seq_tokens, head=0)  # visualize head 0


## 🏗️ Building a Tiny Translation Model with `nn.Transformer`

Below we construct a *very small* encoder–decoder Transformer for English→French number phrases.  
The dataset is intentionally tiny so that **training finishes in under a minute** on a typical laptop CPU/GPU.  


In [ ]:
# -----------------------------------------------------------
# ⚙️  Minimal Transformer for toy translation (EN → FR)
# -----------------------------------------------------------
import torch, math, time
from torch import nn
from torch.utils.data import DataLoader

# 1️⃣ Toy corpus ------------------------------------------------------------
data = [
    ("zero", "zéro"),
    ("one", "un"),
    ("two", "deux"),
    ("three", "trois"),
    ("four", "quatre"),
    ("five", "cinq"),
    ("six", "six"),      # same spelling!
    ("seven", "sept"),
    ("eight", "huit"),
    ("nine", "neuf"),
]

# Build vocabularies
PAD, BOS, EOS = "<pad>", "<bos>", "<eos>"
def build_vocab(sentences):
    vocab = {PAD:0, BOS:1, EOS:2}
    for s in sentences:
        for tok in s.split():
            if tok not in vocab:
                vocab[tok] = len(vocab)
    return vocab

en_vocab = build_vocab([en for en,_ in data])
fr_vocab = build_vocab([fr for _,fr in data])

def encode(sentence, vocab):
    tokens = [BOS] + sentence.split() + [EOS]
    return torch.tensor([vocab[t] for t in tokens], dtype=torch.long)

enc_pairs = [(encode(en, en_vocab), encode(fr, fr_vocab)) for en,fr in data]

# 2️⃣ Dataloader with simple collate ----------------------------------------
def collate(batch):
    src, tgt = zip(*batch)
    src_lens = [len(s) for s in src]
    tgt_lens = [len(t) for t in tgt]
    max_src, max_tgt = max(src_lens), max(tgt_lens)
    src_padded = [torch.cat([s, torch.zeros(max_src-len(s), dtype=torch.long)]) for s in src]
    tgt_padded = [torch.cat([t, torch.zeros(max_tgt-len(t), dtype=torch.long)]) for t in tgt]
    return torch.stack(src_padded), torch.stack(tgt_padded)

loader = DataLoader(enc_pairs, batch_size=10, shuffle=True, collate_fn=collate)

# 3️⃣ Positional Encoding module -------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))
        pe[:,0::2] = torch.sin(pos * div_term)
        pe[:,1::2] = torch.cos(pos * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:,:x.size(1)]

# 4️⃣ Transformer model ----------------------------------------------------
class TinyTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=32, nhead=4, num_layers=2):
        super().__init__()
        self.src_embed = nn.Embedding(len(src_vocab), d_model)
        self.tgt_embed = nn.Embedding(len(tgt_vocab), d_model)
        self.pos_enc  = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=64,
            dropout=0.1,
            batch_first=True
        )
        self.fc_out = nn.Linear(d_model, len(tgt_vocab))

    def forward(self, src, tgt):
        src_mask = None
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)

        src_emb = self.pos_enc(self.src_embed(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt))

        memory = self.transformer.encoder(src_emb, src_key_padding_mask=(src==0))
        output = self.transformer.decoder(
            tgt_emb, memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=(tgt==0),
            memory_key_padding_mask=(src==0))
        logits = self.fc_out(output)
        return logits

# 5️⃣ Training loop --------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyTransformer(en_vocab, fr_vocab).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def shift_right(tensor):
    """Add BOS token at left and drop last token."""
    bos_col = torch.full((tensor.size(0),1), 1, dtype=torch.long)  # BOS=1
    return torch.cat([bos_col.to(tensor.device), tensor[:,:-1]], dim=1)

start = time.time()
for epoch in range(20):       # 20 epochs < 1s on GPU, ~30s CPU
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in  = shift_right(tgt)
        logits  = model(src, tgt_in)
        loss = criterion(logits.view(-1, logits.size(-1)), tgt.view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
end = time.time()
print(f"Training done in {end-start:.2f} s. Final loss: {loss.item():.4f}")
